# Etapa 1

In [99]:
# dados disponíveis em: https://download.inep.gov.br/microdados/microdados_enem_2025.zip
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('ENEM-SPARK-STUDY')
    .master('local[*]')
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "America/Sao_Paulo")
    .getOrCreate()
)

In [100]:
df = spark.read.csv('../data/bronze/RESULTADOS_2025.csv', sep=';', encoding='iso-8859-1', header=True, inferSchema=True)
df.printSchema()

root
 |-- NU_SEQUENCIAL: integer (nullable = true)
 |-- NU_ANO: integer (nullable = true)
 |-- CO_ESCOLA: integer (nullable = true)
 |-- CO_MUNICIPIO_ESC: integer (nullable = true)
 |-- NO_MUNICIPIO_ESC: string (nullable = true)
 |-- CO_UF_ESC: integer (nullable = true)
 |-- SG_UF_ESC: string (nullable = true)
 |-- TP_DEPENDENCIA_ADM_ESC: integer (nullable = true)
 |-- TP_LOCALIZACAO_ESC: integer (nullable = true)
 |-- TP_SIT_FUNC_ESC: integer (nullable = true)
 |-- CO_MUNICIPIO_PROVA: integer (nullable = true)
 |-- NO_MUNICIPIO_PROVA: string (nullable = true)
 |-- CO_UF_PROVA: integer (nullable = true)
 |-- SG_UF_PROVA: string (nullable = true)
 |-- TP_PRESENCA_CN: integer (nullable = true)
 |-- TP_PRESENCA_CH: integer (nullable = true)
 |-- TP_PRESENCA_LC: integer (nullable = true)
 |-- TP_PRESENCA_MT: integer (nullable = true)
 |-- CO_PROVA_CN: integer (nullable = true)
 |-- CO_PROVA_CH: integer (nullable = true)
 |-- CO_PROVA_LC: integer (nullable = true)
 |-- CO_PROVA_MT: integer 

In [101]:
df.count()

4810772

In [102]:
df.describe().show()

+-------+------------------+-------+--------------------+------------------+----------------+------------------+---------+----------------------+-------------------+---------------+------------------+------------------+------------------+-----------+------------------+-------------------+-------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+-----------------+------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------------+--------------------+--------------------+--------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+---------------------+------------------+------------------+------------------+------------------+------------------+------------------+---------------------+------------------+------------------+-------

In [103]:
df.show()

+-------------+------+---------+----------------+-----------------+---------+---------+----------------------+------------------+---------------+------------------+-------------------+-----------+-----------+--------------+--------------+--------------+--------------+-----------+-----------+-----------+-----------+----------+----------+----------+----------+--------------------+--------------------+--------------------+--------------------+---------+--------------------+--------------------+--------------------+--------------------+-----------------+-------------+-------------+-------------+-------------+-------------+---------------+---------------------+-----------+-----------------+-----------------+-----------------+-----------------+-----------------+---------------------+-----------+-----------------+-----------------+-----------------+-----------------+-----------------+---------------------+-----------+-----------------+-----------------+-----------------+-----------------+----

In [104]:
df.select(['TP_PRESENCA_CN'])

DataFrame[TP_PRESENCA_CN: int]

In [105]:
# verificando valores nulos
from pyspark.sql.functions import col, sum, when

columns = ['NU_NOTA_CN','NU_NOTA_CH','NU_NOTA_LC','NU_NOTA_MT','TP_PRESENCA_CN','TP_PRESENCA_CH','TP_PRESENCA_LC','TP_PRESENCA_MT']

df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in columns
]).show()

+----------+----------+----------+----------+--------------+--------------+--------------+--------------+
|NU_NOTA_CN|NU_NOTA_CH|NU_NOTA_LC|NU_NOTA_MT|TP_PRESENCA_CN|TP_PRESENCA_CH|TP_PRESENCA_LC|TP_PRESENCA_MT|
+----------+----------+----------+----------+--------------+--------------+--------------+--------------+
|   1550436|   1353217|   1353217|   1550436|             0|             0|             0|             0|
+----------+----------+----------+----------+--------------+--------------+--------------+--------------+



In [106]:
# verificando valores nulos por aqueles que compareceram no primeiro dia ou seja (1 = presença, 2 = eliminado), pegando ou LC ou CH que são do primeiro dia
from pyspark.sql.functions import col, sum, when

columns = ['NU_NOTA_CN','NU_NOTA_CH','NU_NOTA_LC','NU_NOTA_MT','TP_PRESENCA_CN','TP_PRESENCA_CH','TP_PRESENCA_LC','TP_PRESENCA_MT']

(df
 .filter(col("TP_PRESENCA_CH") == 1)
 .select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in columns])
 .show()
)

+----------+----------+----------+----------+--------------+--------------+--------------+--------------+
|NU_NOTA_CN|NU_NOTA_CH|NU_NOTA_LC|NU_NOTA_MT|TP_PRESENCA_CN|TP_PRESENCA_CH|TP_PRESENCA_LC|TP_PRESENCA_MT|
+----------+----------+----------+----------+--------------+--------------+--------------+--------------+
|    213207|         0|         0|    213207|             0|             0|             0|             0|
+----------+----------+----------+----------+--------------+--------------+--------------+--------------+



In [107]:
# verificando valores nulos por aqueles que compareceram no primeiro dia ou seja (1 = presença, 2 = eliminado), pegando agora CN ou MT que são dos segundo dia
from pyspark.sql.functions import col, sum, when

columns = ['NU_NOTA_CN','NU_NOTA_CH','NU_NOTA_LC','NU_NOTA_MT','TP_PRESENCA_CN','TP_PRESENCA_CH','TP_PRESENCA_LC','TP_PRESENCA_MT']

(df
 .filter(col("TP_PRESENCA_CN") == 1)
 .select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in columns])
 .show()
)

+----------+----------+----------+----------+--------------+--------------+--------------+--------------+
|NU_NOTA_CN|NU_NOTA_CH|NU_NOTA_LC|NU_NOTA_MT|TP_PRESENCA_CN|TP_PRESENCA_CH|TP_PRESENCA_LC|TP_PRESENCA_MT|
+----------+----------+----------+----------+--------------+--------------+--------------+--------------+
|         0|     15988|     15988|         0|             0|             0|             0|             0|
+----------+----------+----------+----------+--------------+--------------+--------------+--------------+



In [108]:
# verificando valores nulos por aqueles que compareceram no primeiro dia ou seja (1 = presença, 2 = eliminado), pegando agora CN ou MT que são dos segundo dia
from pyspark.sql.functions import col, sum, when

columns = ['NU_NOTA_CN','NU_NOTA_CH','NU_NOTA_LC','NU_NOTA_MT','TP_PRESENCA_CN','TP_PRESENCA_CH','TP_PRESENCA_LC','TP_PRESENCA_MT']

(df
 .filter(col("TP_PRESENCA_LC") == 1)
 .count()
)

3457555

Temos um total de 4810772 registros, com a grande maioria sendo inferida através da utilização do inferSchema na leitura do csv.

- Dentre as colunas de notas, temos aproximadamente de 28-31% de valores NULOS, sendo que não temos valores nulos nas colunas de presença.
- Pessoas que compareceram no primeiro dia mas não no segundo: 213207
- Pessoas que compareceram no segundo dia mas não no primeiro: 15988 (que é esperado ser menor mesmo já que candidatos não tem mais chance de passar na grande maioria dos cursos)

-> Ao compararmos os valores de ausência, vemos que bate exatamente com os valores nulos de notas em seus respectivos dias, com o segundo dia tendo mais faltas que o primeiro conforme esperado. Podemos deduzir então que os valores nulos são exatamente as candidatos que faltaram essas provas mas tem inscrição.

# Etapa 2

In [109]:
# Filtrar apenas candidatos da Paraíba (SG_UF_PROVA = 'PB')
from pyspark.sql.functions import col

df_pb = df.filter(col("SG_UF_PROVA") == 'PB')

df_pb.show()

+-------------+------+---------+----------------+----------------+---------+---------+----------------------+------------------+---------------+------------------+--------------------+-----------+-----------+--------------+--------------+--------------+--------------+-----------+-----------+-----------+-----------+----------+----------+----------+----------+--------------------+--------------------+--------------------+--------------------+---------+--------------------+--------------------+--------------------+--------------------+-----------------+-------------+-------------+-------------+-------------+-------------+---------------+---------------------+-----------+-----------------+-----------------+-----------------+-----------------+-----------------+---------------------+-----------+-----------------+-----------------+-----------------+-----------------+-----------------+---------------------+-----------+-----------------+-----------------+-----------------+-----------------+----

In [110]:
# Filtrar apenas candidatos presentes em todas as provas (TP_PRESENCA_* = 1)
from pyspark.sql.functions import col

df_pb_presenca = (df_pb
 .filter(
    (col("TP_PRESENCA_CN") == 1) &
    (col("TP_PRESENCA_CH") == 1) &
    (col("TP_PRESENCA_LC") == 1) &
    (col("TP_PRESENCA_MT") == 1)
))

df_pb_presenca.show()

+-------------+------+---------+----------------+----------------+---------+---------+----------------------+------------------+---------------+------------------+--------------------+-----------+-----------+--------------+--------------+--------------+--------------+-----------+-----------+-----------+-----------+----------+----------+----------+----------+--------------------+--------------------+--------------------+--------------------+---------+--------------------+--------------------+--------------------+--------------------+-----------------+-------------+-------------+-------------+-------------+-------------+---------------+---------------------+-----------+-----------------+-----------------+-----------------+-----------------+-----------------+---------------------+-----------+-----------------+-----------------+-----------------+-----------------+-----------------+---------------------+-----------+-----------------+-----------------+-----------------+-----------------+----

In [111]:
display(df.count())
display(df.count() - df_pb.count())
display(df_pb.count())
display(df_pb.count() - df_pb_presenca.count())
display(df_pb_presenca.count())

4810772

4668737

142035

41152

100883

Aqui vemos que tivemos 4668737 registros descartados no primeiro filtro por conta de termos selecionado apenas um dos 27 estados, sendo que a Paraíba não é um dos estados com maiores quantidades de pessoas (tipo São Paulo).

Enquanto isso, no filtro de presença, vemos que corresponde a (41152/142035) * 100 que é praticamente 29% das pessoas que faltaram a prova, correspondendo a nosso intervalo geral de 28-31% de pessoas que faltaram somando todos os estados.

In [112]:
# Criar coluna NOTA_MEDIA
from pyspark.sql.functions import col

df = df.withColumn('NOTA_MEDIA', (
    col('NU_NOTA_CH') + 
    col('NU_NOTA_LC') + 
    col('NU_NOTA_MT') + 
    col('NU_NOTA_CN')
    )
    /4
)

df.select('NU_SEQUENCIAL', 'NOTA_MEDIA').show()

+-------------+------------------+
|NU_SEQUENCIAL|        NOTA_MEDIA|
+-------------+------------------+
|       206403| 584.1999999999999|
|      3604651|              NULL|
|      1461268|           413.875|
|      4301058|              NULL|
|      3148322|             540.0|
|      1035588|             516.3|
|      3587454|            426.95|
|       140768|             549.5|
|      3541867|             621.0|
|      3371353|           433.825|
|       377894|             434.8|
|      3446319|              NULL|
|       134650|             545.8|
|      1620924|              NULL|
|       588106|              NULL|
|      1399198|424.07500000000005|
|       941795|           458.375|
|      1894153|379.27500000000003|
|      2262755|              NULL|
|      3575012|           588.125|
+-------------+------------------+
only showing top 20 rows


In [113]:
from pyspark.sql.functions import col, when

df = df.withColumn(
    "FAIXA_DESEMPENHO",
    when(col("NOTA_MEDIA").isNull(), None)
    .when(col("NOTA_MEDIA") < 400, "ABAIXO DO BÁSICO")
    .when(col("NOTA_MEDIA") < 550, "BÁSICO")
    .when(col("NOTA_MEDIA") < 700, "ADEQUADO")
    .otherwise("AVANÇADO")
)

df.select("NOTA_MEDIA", "FAIXA_DESEMPENHO").show()

+------------------+----------------+
|        NOTA_MEDIA|FAIXA_DESEMPENHO|
+------------------+----------------+
| 584.1999999999999|        ADEQUADO|
|              NULL|            NULL|
|           413.875|          BÁSICO|
|              NULL|            NULL|
|             540.0|          BÁSICO|
|             516.3|          BÁSICO|
|            426.95|          BÁSICO|
|             549.5|          BÁSICO|
|             621.0|        ADEQUADO|
|           433.825|          BÁSICO|
|             434.8|          BÁSICO|
|              NULL|            NULL|
|             545.8|          BÁSICO|
|              NULL|            NULL|
|              NULL|            NULL|
|424.07500000000005|          BÁSICO|
|           458.375|          BÁSICO|
|379.27500000000003|ABAIXO DO BÁSICO|
|              NULL|            NULL|
|           588.125|        ADEQUADO|
+------------------+----------------+
only showing top 20 rows


In [ ]:
df_silver = df.select(
    'NO_MUNICIPIO_ESC', # nome do municipio escolar
    'SG_UF_ESC', # sigla da uf da escola
    ''
)

In [114]:
df.explain()

== Physical Plan ==
*(1) Project [NU_SEQUENCIAL#28956, NU_ANO#28957, CO_ESCOLA#28958, CO_MUNICIPIO_ESC#28959, NO_MUNICIPIO_ESC#28960, CO_UF_ESC#28961, SG_UF_ESC#28962, TP_DEPENDENCIA_ADM_ESC#28963, TP_LOCALIZACAO_ESC#28964, TP_SIT_FUNC_ESC#28965, CO_MUNICIPIO_PROVA#28966, NO_MUNICIPIO_PROVA#28967, CO_UF_PROVA#28968, SG_UF_PROVA#28969, TP_PRESENCA_CN#28970, TP_PRESENCA_CH#28971, TP_PRESENCA_LC#28972, TP_PRESENCA_MT#28973, CO_PROVA_CN#28974, CO_PROVA_CH#28975, CO_PROVA_LC#28976, CO_PROVA_MT#28977, NU_NOTA_CN#28978, NU_NOTA_CH#28979, NU_NOTA_LC#28980, ... 47 more fields]
+- *(1) Project [NU_SEQUENCIAL#28956, NU_ANO#28957, CO_ESCOLA#28958, CO_MUNICIPIO_ESC#28959, NO_MUNICIPIO_ESC#28960, CO_UF_ESC#28961, SG_UF_ESC#28962, TP_DEPENDENCIA_ADM_ESC#28963, TP_LOCALIZACAO_ESC#28964, TP_SIT_FUNC_ESC#28965, CO_MUNICIPIO_PROVA#28966, NO_MUNICIPIO_PROVA#28967, CO_UF_PROVA#28968, SG_UF_PROVA#28969, TP_PRESENCA_CN#28970, TP_PRESENCA_CH#28971, TP_PRESENCA_LC#28972, TP_PRESENCA_MT#28973, CO_PROVA_CN#28974

Nesse explain, o spark está lendo o csv, depois criando a nova coluna NOTA_MEDIA, e logo em seguida criando outra coluna FAIXA_DESEMPENHO

- tive que instalar o UV primeiro na minha máquina, depois usei o uv add com as dependencias, logo em seguida instalei o java 17 com openjdk
- decidi usar shuffle com 8 partições por ser algo mais local 
- da primeira vez que li o csv ele só identificou uma coluna por conta de eu não ter usado o ; como separador de colunas
- após ajeitar o ;, por ser um csv, ele só leu como string o tipo das colunas, daí foi necessário ajeitar essa tipagem com inferSchema = True na leitura
- foi necessário inserir o encoding em latin (iso8859-1) além de header=True para identificar esse cabeçalho
- ao tentar fazer a parte da faixa_desempenho usando UDF obtive erro de timeout no worker python, tive dificuldade nessa parte então fiz com when e otherwise para otimizar com catalyst